In [1]:
import sys
import time

import arviz as az
import numpy as np
from regina import Triangulation3
from tqdm import tqdm

sys.path.append("./src/")
from functions3d import choosemove
from functions3d_fast_accept_all import choosemove as choosemove_fast

In [2]:
alph = 'abcdefghijklmnopqrstuvwxyz'
conv = {alph[i]: i for i in range(len(alph))}

In [3]:
gamma = 1 / 5

isos_slow = ["cMcabbgqs"]

t = Triangulation3.fromIsoSig(isos_slow[0])  # type: ignore
f = t.fVector()

its = 100_000
slow_esss = []

start = time.perf_counter()
for c in tqdm(range(20)):
    for i in range(its):
        res = choosemove(isos_slow[-1], f, gamma)
        assert res
        iso, f = res
        isos_slow.append(iso)

    slow_degree = np.array([conv[iso[0]] for iso in isos_slow])
    slow_ess = az.ess(slow_degree) / len(isos_slow)
    slow_esss.append(slow_ess)
slow_time = time.perf_counter() - start

100%|██████████| 20/20 [16:00<00:00, 48.04s/it]


In [4]:
gamma = 1 / 5

isos = ["cMcabbgqs"]

f = None
aut = None

its = 100_000
fast_esss = []

start = time.perf_counter()
for c in tqdm(range(20)):
    for i in range(its):
        iso, f, aut = choosemove_fast(
            isos[-1], gamma, f=f, aut=aut
        )
        isos.append(iso)

    fast_degree = np.array([conv[iso[0]] for iso in isos])
    fast_ess = az.ess(fast_degree) / len(isos)
    fast_esss.append(fast_ess)
fast_time = time.perf_counter() - start

100%|██████████| 20/20 [04:46<00:00, 14.32s/it]


In [5]:
# ESS of the number of tetrahedra, per step and per second of wall time.
slow_ess_total = az.ess(slow_degree)
fast_ess_total = az.ess(fast_degree)

print(f"original:   ESS/step {slow_ess_total / len(isos_slow):.4f}, ESS/s {slow_ess_total / slow_time:.1f}, {slow_time:.0f}s")
print(f"accept-all: ESS/step {fast_ess_total / len(isos):.4f}, ESS/s {fast_ess_total / fast_time:.1f}, {fast_time:.0f}s")
print(f"speedup in ESS/s: {(fast_ess_total / fast_time) / (slow_ess_total / slow_time):.1f}x")

original:   ESS/step 0.0003, ESS/s 0.7, 961s
accept-all: ESS/step 0.0003, ESS/s 2.0, 286s
speedup in ESS/s: 3.0x
